### Graph Neural networks with PyTorch

https://www.geeksforgeeks.org/deep-learning/graph-neural-networks-with-pytorch/

In [15]:
import torch
from torch_geometric.data import Data
from torch_geometric.datasets import Planetoid
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import os

# for plotting
import matplotlib.pyplot as plt

# Loading the Cora dataset
dataset = Planetoid(root='data/Planetoid', name='Cora')

Defining GNN model using pytorch

In [16]:
# structure of GNN model 
class CustomGNN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(CustomGNN, self).__init__()
        self.layer1 = GCNConv(input_dim, hidden_dim)
        self.layer2 = GCNConv(hidden_dim, output_dim)
# forward pass 
    def forward(self, feature_data, edge_info):
        # First GCN layer
        x = self.layer1(feature_data, edge_info)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        # Second GCN layer
        x = self.layer2(x, edge_info)
        return F.log_softmax(x, dim=1)

# Initialize the GNN model
input_features = dataset.num_node_features
num_classes = dataset.num_classes
model = CustomGNN(input_features, 16, num_classes)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

graph_data = dataset[0]  # Get the graph data

Training the model. Initialise model, define optimiser, running training loop.

In [17]:
def train_model():
    model.train()
    optimizer.zero_grad()
    output = model(graph_data.x, graph_data.edge_index)
    loss = F.nll_loss(output[graph_data.train_mask], graph_data.y[graph_data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(200):
    loss_value = train_model()
    print(f'Epoch: {epoch+1:03d}, Loss: {loss_value:.4f}')

Epoch: 001, Loss: 1.9511
Epoch: 002, Loss: 1.8468
Epoch: 003, Loss: 1.7275
Epoch: 004, Loss: 1.5677
Epoch: 005, Loss: 1.4585
Epoch: 006, Loss: 1.2906
Epoch: 007, Loss: 1.1649
Epoch: 008, Loss: 1.0363
Epoch: 009, Loss: 0.9223
Epoch: 010, Loss: 0.8154
Epoch: 011, Loss: 0.6864
Epoch: 012, Loss: 0.6321
Epoch: 013, Loss: 0.5272
Epoch: 014, Loss: 0.4695
Epoch: 015, Loss: 0.3718
Epoch: 016, Loss: 0.3791
Epoch: 017, Loss: 0.3221
Epoch: 018, Loss: 0.2687
Epoch: 019, Loss: 0.2719
Epoch: 020, Loss: 0.2359
Epoch: 021, Loss: 0.1873
Epoch: 022, Loss: 0.2068
Epoch: 023, Loss: 0.1574
Epoch: 024, Loss: 0.1455
Epoch: 025, Loss: 0.1393
Epoch: 026, Loss: 0.0988
Epoch: 027, Loss: 0.1061
Epoch: 028, Loss: 0.1119
Epoch: 029, Loss: 0.0958
Epoch: 030, Loss: 0.1071
Epoch: 031, Loss: 0.0739
Epoch: 032, Loss: 0.0831
Epoch: 033, Loss: 0.0881
Epoch: 034, Loss: 0.0611
Epoch: 035, Loss: 0.0405
Epoch: 036, Loss: 0.0620
Epoch: 037, Loss: 0.0837
Epoch: 038, Loss: 0.0802
Epoch: 039, Loss: 0.0651
Epoch: 040, Loss: 0.0737


4. Evaluating models performance. 

In [18]:
# nothing to plot
def evaluate_model():
    model.eval()
    with torch.no_grad():
        predictions = model(graph_data.x, graph_data.edge_index).argmax(dim=1)
        correct = (predictions[graph_data.test_mask] == graph_data.y[graph_data.test_mask]).sum()
        acc = int(correct) / int(graph_data.test_mask.sum())

        print(f'length of predictions: {predictions.shape}')
        return acc

accuracy = evaluate_model()
print(f'Test Accuracy: {accuracy:.4f}')

length of predictions: torch.Size([2708])
Test Accuracy: 0.7990


### Message passing for ff_script

In [19]:
from ff_script import *

Using device: cpu


In [20]:
"""#-------------------------------GRAPH NN MODEL -------------------------------
# 1. get node features, first row of the samples array 
x = samples[0,:]

# 2. get edge features, second row of the samples array 
edge_index = samples[1,:]

# 3. get triplet features
triplet = samples[2,:]

# 4. create pytorch geometric data object
data = Data(x=x, edge_index=edge_index)
"""

'#-------------------------------GRAPH NN MODEL -------------------------------\n# 1. get node features, first row of the samples array \nx = samples[0,:]\n\n# 2. get edge features, second row of the samples array \nedge_index = samples[1,:]\n\n# 3. get triplet features\ntriplet = samples[2,:]\n\n# 4. create pytorch geometric data object\ndata = Data(x=x, edge_index=edge_index)\n'

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


### Obtaining outputs to put into message passing algorithm

In [22]:
from ff_script import generate_molecules # for generating molecules
from ff_script import molecules_to_tensors # converting molecules to tensors
from ff_script import load_molecules_from_xyz # reading molecule info from .xyz file

In [23]:
molecules = generate_molecules(n_h2o=1, n_hf=0, filename='one_h20.xyz')
# first item: x y z coordinates for each atom 
# second item: atoms 
# third item: molecular energy 
# fourth item: force per atom 
molecules

Generated 1 structures → one_h20.xyz


[(array([[-0.03545485, -0.05319379, -0.02308166],
         [-0.19478466, -0.77879958, -0.14722355],
         [ 0.30920392,  0.72416068, -0.10541983]]),
  ['O', 'H', 'H'],
  np.float64(1.140304065568201),
  array([[  2.46694888,  14.01193393,   3.1520891 ],
         [ -3.67246439, -16.72897064,  -2.86324318],
         [  1.20551551,   2.7170367 ,  -0.28884593]]))]

In [24]:
# gets all features for GNN
features = molecules_to_tensors(molecules, device)
print(features)
# first item: node tensors, NODE FEATURES O, H, H
# second item: edge tensors, EDGE FEATURES 
# third item: triplet tensor, TRIPLET FEATURES 
# fourth item: positive tensor, positions of atoms in x y z 
# fifth item: total energy
# sixth item: force per atom
# number of elements 
# target energy 

([(tensor([[1.9459],
        [0.6931],
        [0.6931]]), tensor([[2.6391, 0.5673]]), tensor([[0.]]), tensor([[-0.0355, -0.0532, -0.0231],
        [-0.1948, -0.7788, -0.1472],
        [ 0.3092,  0.7242, -0.1054]], requires_grad=True), np.float64(1.140304065568201), tensor([[  2.4669,  14.0119,   3.1521],
        [ -3.6725, -16.7290,  -2.8632],
        [  1.2055,   2.7170,  -0.2888]]), 3)], [np.float64(1.140304065568201)])


In [25]:
# features of each node
node_features = features[0][0][0]
node_features

tensor([[1.9459],
        [0.6931],
        [0.6931]])

In [26]:
# edge features
edge_features = features[0][0][1]
edge_features

tensor([[2.6391, 0.5673]])

In [27]:
# triplet features
trip_features = features[0][0][2]
trip_features

tensor([[0.]])

In [28]:
# position features
pos_features = features[0][0][3]
pos_features

tensor([[-0.0355, -0.0532, -0.0231],
        [-0.1948, -0.7788, -0.1472],
        [ 0.3092,  0.7242, -0.1054]], requires_grad=True)

In [29]:
# total energy
E_total = features[0][0][4]
E_total

np.float64(1.140304065568201)

In [30]:
# force per atom
F_atom = features[0][0][5]
F_atom

tensor([[  2.4669,  14.0119,   3.1521],
        [ -3.6725, -16.7290,  -2.8632],
        [  1.2055,   2.7170,  -0.2888]])

In [31]:
# number of elements
N = features[0][0][6]
N

3

In [32]:
# number of elements
target_E = features[1]


In [33]:
# printing whats in one_h20.xyz
load_molecules_from_xyz('one_h20.xyz')

Loaded 1 structures from one_h20.xyz


[(array([[-0.035455, -0.053194, -0.023082],
         [-0.194785, -0.7788  , -0.147224],
         [ 0.309204,  0.724161, -0.10542 ]]),
  ['O', 'H', 'H'],
  np.float64(0.1140297922993285),
  array([[  2.46694612,  14.01188063,   3.15207393],
         [ -3.67245029, -16.72889175,  -2.86323148],
         [  1.20550417,   2.71701112,  -0.28884245]]))]

## Message passing algorithm

https://medium.com/@m.nusret.ozates/pytorch-geometric-basics-how-message-passing-works-677dd635ea0f

In [55]:
from typing import Optional

import torch
from torch import Tensor
from torch.nn import Linear, Parameter
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
from torch_geometric.data import Data

In [56]:
class GCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.empty(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]
        print("Forward pass...")
        print(f"x shape: {x.shape}")
        print(f"edge_index shape: {edge_index.shape}")

        # Step 1: Add self-loops to the adjacency matrix.
        # means that a node's own features influence the result
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Step 2: Linearly transform node feature matrix.
        x = self.lin(x)

        # Step 3: Compute normalization.
        source, target = edge_index
        deg = degree(target, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[source] * deg_inv_sqrt[target]

        # Step 4-5: Start propagating messages.
        out = self.propagate(edge_index, x=x, norm=norm)

        # Step 6: Apply a final bias vector.
        out = out + self.bias

        return out

    def message(self, x_i, x_j, norm, edge_index):
        # x_j has shape [E, out_channels]
        print("Creating messages...")
        print(f"x_i shape: {x_i.shape}")
        print(f"x_i : {x_i}")
        
        print(f"x_j shape: {x_j.shape}")
        print(f"norm shape: {norm.shape}")

        # Step 4: Normalize node features.
        return norm.view(-1, 1) * x_j

    def aggregate(
        self,
        inputs: Tensor,
        index: Tensor,
        ptr: Optional[Tensor] = None,
        dim_size: Optional[int] = None,
    ) -> Tensor:
        print("Aggregating messages...")
        print(f"Inputs shape: {inputs.shape}")
        print(f"Index shape: {index.shape}")
        print(index)
        return super().aggregate(inputs, index, ptr, dim_size)

    def update(self, inputs: Tensor) -> Tensor:
        print("Updating node embeddings...")
        print(f"Inputs shape: {inputs.shape}")
        print(inputs)
        return super().update(inputs)

#--------------------------------------------------

# need edge_index and information about each edge? 
edge_index = torch.tensor([[0, 1], # edge from node 0 to node 1
                           [1, 0], # edge from node 1 to node 0
                           [1, 2], # edge from node 1 to node 2
                           [2, 1]], # edge from node 2 to node 1
                           dtype=torch.long)

x = node_features #
# how to add in information about triplets 
#--------------------------------------------------

data = Data(x=x, edge_index=edge_index.t().contiguous())

conv = GCNConv(1, 2)
out = conv(data.x, data.edge_index)
print(out)

Forward pass...
x shape: torch.Size([3, 1])
edge_index shape: torch.Size([2, 4])
Creating messages...
x_i shape: torch.Size([7, 2])
x_i : tensor([[-0.6750, -0.0373],
        [-1.8950, -0.1046],
        [-0.6750, -0.0373],
        [-0.6750, -0.0373],
        [-1.8950, -0.1046],
        [-0.6750, -0.0373],
        [-0.6750, -0.0373]], grad_fn=<IndexSelectBackward0>)
x_j shape: torch.Size([7, 2])
norm shape: torch.Size([7])
Aggregating messages...
Inputs shape: torch.Size([7, 2])
Index shape: torch.Size([7])
tensor([1, 0, 2, 1, 0, 1, 2])
Updating node embeddings...
Inputs shape: torch.Size([3, 2])
tensor([[-1.2231, -0.0675],
        [-1.2742, -0.0703],
        [-0.6131, -0.0338]], grad_fn=<ScatterAddBackward0>)
tensor([[-1.2231, -0.0675],
        [-1.2742, -0.0703],
        [-0.6131, -0.0338]], grad_fn=<AddBackward0>)


In [ ]:
from typing import Optional

import torch
from torch import Tensor
from torch.nn import Linear, Parameter
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
from torch_geometric.data import Data

In [57]:
class GCNConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.empty(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]
        print("Forward pass...")
        print(f"x shape: {x.shape}")
        print(f"edge_index shape: {edge_index.shape}")

        # Step 1: Add self-loops to the adjacency matrix.
        # means that a node's own features influence the result
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Step 2: Linearly transform node feature matrix.
        x = self.lin(x)

        # Step 3: Compute normalization.
        source, target = edge_index
        deg = degree(target, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[source] * deg_inv_sqrt[target]

        # Step 4-5: Start propagating messages.
        out = self.propagate(edge_index, x=x, norm=norm)

        # Step 6: Apply a final bias vector.
        out = out + self.bias

        return out

    def message(self, x_i, x_j, norm, edge_index):
        # x_j has shape [E, out_channels]
        print("Creating messages...")
        print(f"x_i shape: {x_i.shape}")
        print(f"x_i : {x_i}")
        
        print(f"x_j shape: {x_j.shape}")
        print(f"norm shape: {norm.shape}")

        # Step 4: Normalize node features.
        return norm.view(-1, 1) * x_j

    def aggregate(
        self,
        inputs: Tensor,
        index: Tensor,
        ptr: Optional[Tensor] = None,
        dim_size: Optional[int] = None,
    ) -> Tensor:
        print("Aggregating messages...")
        print(f"Inputs shape: {inputs.shape}")
        print(f"Index shape: {index.shape}")
        print(index)
        return super().aggregate(inputs, index, ptr, dim_size)

    def update(self, inputs: Tensor) -> Tensor:
        print("Updating node embeddings...")
        print(f"Inputs shape: {inputs.shape}")
        print(inputs)
        return super().update(inputs)

# describes the combination of nodes 
edge_index = torch.tensor([[0, 1], # edge from node 0 to node 1
                           [1, 0], # edge from node 1 to node 0
                           [1, 2], # edge from node 1 to node 2
                           [2, 1]], # edge from node 2 to node 1
                           dtype=torch.long)
x = torch.tensor([[-1], [0], [1]], dtype=torch.float)

data = Data(x=x, edge_index=edge_index.t().contiguous())

conv = GCNConv(1, 2)
out = conv(data.x, data.edge_index)
print(out)

Forward pass...
x shape: torch.Size([3, 1])
edge_index shape: torch.Size([2, 4])
Creating messages...
x_i shape: torch.Size([7, 2])
x_i : tensor([[ 0.0000, -0.0000],
        [-0.7892,  0.1310],
        [ 0.7892, -0.1310],
        [ 0.0000, -0.0000],
        [-0.7892,  0.1310],
        [ 0.0000, -0.0000],
        [ 0.7892, -0.1310]], grad_fn=<IndexSelectBackward0>)
x_j shape: torch.Size([7, 2])
norm shape: torch.Size([7])
Aggregating messages...
Inputs shape: torch.Size([7, 2])
Index shape: torch.Size([7])
tensor([1, 0, 2, 1, 0, 1, 2])
Updating node embeddings...
Inputs shape: torch.Size([3, 2])
tensor([[-0.3946,  0.0655],
        [ 0.0000,  0.0000],
        [ 0.3946, -0.0655]], grad_fn=<ScatterAddBackward0>)
tensor([[-0.3946,  0.0655],
        [ 0.0000,  0.0000],
        [ 0.3946, -0.0655]], grad_fn=<AddBackward0>)


source to target: <br>
source = neighbours <br>
target = self

## Following tutorial

https://www.kaggle.com/code/suvroo/gnn-from-scratch-trial

In [2]:
import numpy as np
from scipy.linalg import sqrtm 
from scipy.special import softmax
import networkx as nx
from networkx.algorithms.community.modularity_max import greedy_modularity_communities
import matplotlib.pyplot as plt
from matplotlib import animation
%matplotlib inline
from IPython.display import HTML

In [3]:
# A = adjacency matrix 
A = np.array([[0, 1 ,1],
              [1, 0, 0],
              [1, 0, 0]])

# creating graph from adjacency matrix 
g = nx.from_numpy_array(A)


In [4]:
A_mod = A + np.eye(g.number_of_nodes())

In [5]:
# D for A_mod:
D_mod = np.zeros_like(A_mod)
np.fill_diagonal(D_mod, A_mod.sum(axis=1).flatten())

# Inverse square root of D:
D_mod_invroot = np.linalg.inv(sqrtm(D_mod))
D_mod

array([[3., 0., 0.],
       [0., 2., 0.],
       [0., 0., 2.]])

In [6]:
D_mod_invroot

array([[0.57735027, 0.        , 0.        ],
       [0.        , 0.70710678, 0.        ],
       [0.        , 0.        , 0.70710678]])

In [34]:
node_labels = {i: i+1 for i in range(g.number_of_nodes())}
# pos = nx.planar_layout(g) // original

# using pos from pos_features from ff_script.py
pos = pos_features.detach().numpy()[:-1].T

In [35]:
fig, ax = plt.subplots(figsize=(10,10))
nx.draw(
    g, pos, with_labels=True, 
    labels=node_labels, 
    node_color='#83C167', 
    ax=ax, edge_color='gray', node_size=1500, font_size=30, font_family='serif'
)
plt.savefig('simple_graph.png', bbox_inches='tight', transparent=True)

In [36]:
pos 

array([[-0.03545485, -0.19478466],
       [-0.05319379, -0.7787996 ],
       [-0.02308166, -0.14722355]], dtype=float32)

In [37]:
A_hat = D_mod_invroot @ A_mod @ D_mod_invroot

In [38]:
H = np.zeros((g.number_of_nodes(), 1))
H[0,0] = 1 # the "water drop"
iters = 10
results = [H.flatten()]
for i in range(iters):
    H = A_hat @ H
    results.append(H.flatten())

In [39]:
print(f"Initial signal input: {results[0]}")
print(f"Final signal output after running {iters} steps of message-passing:  {results[-1]}")

Initial signal input: [1. 0. 0.]
Final signal output after running 10 steps of message-passing:  [0.42857144 0.3499271  0.3499271 ]


In [40]:
fig, ax = plt.subplots(figsize=(10, 10))

kwargs = {'cmap': 'hot', 'node_size': 1500, 'edge_color': 'gray', 
          'vmin': np.array(results).min(), 'vmax': np.array(results).max()*1.1}

def update(idx):
    ax.clear()
    colors = results[idx]
    nx.draw(g, pos, node_color=colors, ax=ax, **kwargs)
    ax.set_title(f"Iter={idx}", fontsize=20)

anim = animation.FuncAnimation(fig, update, frames=len(results), interval=1000, repeat=True)

In [41]:
anim.save(
    'water_drop.mp4', 
    dpi=600, bitrate=-1,
    savefig_kwargs={'transparent': True, 'facecolor': 'none'},
)
HTML(anim.to_html5_video())

MovieWriter ffmpeg unavailable; using Pillow instead.


IndexError: list index out of range